# Phase 4: Causal Patching with nnsight

This notebook will take your cleanly labeled pairs (clean vs counterfactual) and use `nnsight` to perform surgical intervention on Layer 10 of Llama-3.2-3B. We will prove that injecting the hallucinating hidden state causally forces the model to derail.

In [ ]:
!pip install -q nnsight transformers accelerate
import huggingface_hub
huggingface_hub.login(token="YOUR_HF_TOKEN") # REPLACE WITH YOUR TOKEN

## 1. Load Model and Data

In [ ]:
import json
import torch
from nnsight import LanguageModel
from transformers import AutoTokenizer

print("Loading Llama-3.2-3B via nnsight...")
model = LanguageModel("meta-llama/Llama-3.2-3B-Instruct", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print("Downloading dataset from Hugging Face...")
!wget -q https://huggingface.co/datasets/AnishRacherla/LinguaFranca-Phase3/resolve/main/data/processed/2wikimultihopqa/augmented.jsonl -O augmented.jsonl

with open("augmented.jsonl") as f:
    data = [json.loads(line) for line in f]
    
counterfactuals = [d for d in data if d.get("is_counterfactual", False)]
cleans = {d["id"]: d for d in data if not d.get("is_counterfactual", False)}

print(f"Loaded {len(cleans)} clean examples and {len(counterfactuals)} counterfactuals.")

## 2. Prepare Perfect Minimal Pairs

In [ ]:
# Find pairs where clean succeeds and CF fails
pairs = []
for cf in counterfactuals:
    clean = cleans.get(cf["clean_pair_id"])
    if not clean: continue
    
    clean_h1 = next((h for h in clean.get("hops", []) if h["hop_idx"] == 1), None)
    cf_h1 = next((h for h in cf.get("hops", []) if h["hop_idx"] == 1), None)
    
    if clean_h1 and cf_h1 and clean_h1.get("label") == 0 and cf_h1.get("label") == 1:
        pairs.append((clean, cf))

print(f"Found {len(pairs)} perfect minimal pairs for causal patching.")

## 3. Perform Causal Brain Surgery!

In [ ]:
target_layer = 10 # From Phase 3 results!
clean, cf = pairs[0]

clean_prompt = clean["prompt"]
cf_prompt = cf["prompt"]

print("=== Clean Question ===")
print(clean["question"])
print("\n=== Counterfactual Question ===")
print(cf["question"])

# We will run both through the model, and intervene on Layer 10!
with model.trace() as tracer:
    # 1. Trace the counterfactual to harvest its hallucinating brain waves
    with tracer.invoke(cf_prompt):
        # Save the hidden state at layer 10 for the LAST token of the prompt
        cf_hidden_state = model.model.layers[target_layer].output[0][:, -1, :].save()
        
    # 2. Trace the clean prompt and inject the hallucinating brain waves!
    with tracer.invoke(clean_prompt):
        # Overwrite the clean model's Layer 10 hidden state at the last token
        model.model.layers[target_layer].output[0][:, -1, :] = cf_hidden_state
        
        # Generate the corrupted output
        corrupted_output = model.generator(max_new_tokens=100)

print("\n=== Corrupted Generation ===")
print(tokenizer.decode(corrupted_output.save().value[0][-100:], skip_special_tokens=True))